In [20]:
import sys
sys.path.append("../")
import torch 
from torch import nn
import src.config as config
import pandas as pd
from collections import Counter

train = pd.read_csv(config.TRAIN_DATA_PATH)
test = pd.read_csv(config.TEST_DATA_PATH)

train["language"].value_counts()
test["language"].value_counts()

language
en    274
es    170
de    161
pt    101
fr     94
Name: count, dtype: int64

In [2]:
train["priority"].value_counts()

priority
high      1343
medium    1233
low        624
Name: count, dtype: int64

In [3]:
train.head()

,context_problem,answer,type,queue,priority,language,business_type,tags,cleaned_context,stemmed_context
0,Inquiry about Return and Exchange Policy for E...,"Dear <name>,\n\nThank you for contacting the T...",Request,Returns and Exchanges,low,en,Tech Online Store,"['Returns and Exchanges', 'Customer Service', ...",inquiry about return and exchange policy for e...,inquiri about return and exchang polici for ep...
1,Solicitud de intercambio de impresora Estimado...,"Estimado <name>,\n\nGracias por ponerte en con...",Request,Returns and Exchanges,medium,es,Tech Online Store,"['Returns and Exchanges', 'Product Support', '...",solicitud de intercambio de impresora estimado...,solicitud de intercambi de impresor estim sopo...
2,"Problem mit Netzteil GS108 Hallo,\n\nich erleb...","Hallo <name>, um Ihr GS108-Switch zu beheben: ...",Incident,Product Support,medium,de,Tech Online Store,"['Technical Support', 'Hardware Failure', 'Net...",problem mit netzteil gs108 hallo ich erlebe hä...,probl mit netzteil gs108 hallo ich erleb haufi...
3,Asistencia para compartir pantalla en Zoom Hol...,"Hola,\n\nPara configurar el uso compartido de ...",Request,Technical Support,low,es,IT Services,"['Technical Support', 'Product Support', 'Soft...",asistencia para compartir pantalla en zoom hol...,asistent par compart pantall en zoom hol neces...
4,Kunde benötigt den Dunkelmodus und Erweiterung...,"Hallo <name>,\n\num den Dunkelmodus in VS Code...",Change,Product Support,low,de,Software Development Company,"['Technical Support', 'Product Support', 'Feat...",kunde benötigt den dunkelmodus und erweiterung...,kund benotigt den dunkelmodus und erweiter fur...


In [4]:
print(train.shape, test.shape)

(3200, 10) (800, 10)


In [5]:
print(train[["cleaned_context", "queue"]].isna().sum())
print(test[["cleaned_context", "queue"]].isna().sum())

cleaned_context    0
queue              0
dtype: int64
cleaned_context    0
queue              0
dtype: int64


In [6]:
print(train["queue"].value_counts())

queue
Technical Support                  1053
Product Support                     552
Customer Service                    502
IT Support                          356
Billing and Payments                270
Returns and Exchanges               158
Service Outages and Maintenance     113
Sales and Pre-Sales                 110
General Inquiry                      44
Human Resources                      42
Name: count, dtype: int64


In [7]:
print(test["queue"].value_counts())

queue
Technical Support                  264
Product Support                    138
Customer Service                   125
IT Support                          89
Billing and Payments                68
Returns and Exchanges               39
Service Outages and Maintenance     28
Sales and Pre-Sales                 27
General Inquiry                     11
Human Resources                     11
Name: count, dtype: int64


In [8]:
print(train["queue"].nunique(), sorted(train["queue"].unique()))
print(test["queue"].nunique(), sorted(test["queue"].unique()))

10 ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
10 ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']


In [ ]:
# Check the distribution of token lengths in the cleaned_context column to determine an appropriate max_length for tokenization.
token_lengths = train["cleaned_context"].apply(lambda text: len(text.split()))
print(token_lengths.describe(percentiles=[0.90, 0.95, 0.99]))

count    3200.000000
mean      114.358437
std        71.038454
min         2.000000
90%       212.000000
95%       242.050000
99%       305.010000
max       401.000000
Name: cleaned_context, dtype: float64


In [28]:
counter = Counter()

for text in train["cleaned_context"]:
    counter.update(text.split())
print(counter.most_common(20))
print(len(counter))

[('de', 10944), ('to', 4404), ('a', 3939), ('the', 3424), ('que', 3024), ('i', 2941), ('para', 2924), ('and', 2498), ('this', 2407), ('la', 2340), ('your', 2220), ('you', 2156), ('for', 2079), ('ich', 1912), ('our', 1877), ('en', 1843), ('zu', 1796), ('in', 1761), ('support', 1695), ('und', 1570)]
16391


In [29]:
for text in train["cleaned_context"]:
    counter.update(text.split())

total_tokens = sum(counter.values())
print(f"Total tokens: {total_tokens}")

for vocab_size in [5000, 10000, 15000, 16391]:
    top_tokens = counter.most_common(vocab_size)
    covered = sum(count for _, count in top_tokens)
    coverage = covered / total_tokens * 100
    print(f"vocab_size: {vocab_size}, coverage: {coverage:.2f}%")

Total tokens: 731894
vocab_size: 5000, coverage: 94.18%
vocab_size: 10000, coverage: 98.21%
vocab_size: 15000, coverage: 99.62%
vocab_size: 16391, coverage: 100.00%
